In [1]:
!pip install kagglehub
!pip install tiktoken
!pip install tqdm
!pip install datasets
!git clone https://github.com/tylerksoong/transformers.git
!cp -r transformers/transformer/ ./

fatal: destination path 'transformers' already exists and is not an empty directory.


In [2]:

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
import pandas as pd
import copy
from tqdm import tqdm
import tiktoken
import kagglehub
import gc

PAD_ID = 50257
DEVICE = "cuda"

In [3]:
from datasets import load_dataset

train_raw = load_dataset("roneneldan/TinyStories", split="train[:1%]")
valid_raw = load_dataset("roneneldan/TinyStories", split="validation[:1%]")

In [4]:
print(train_raw)

Dataset({
    features: ['text'],
    num_rows: 21197
})


In [5]:
class TinyStories(Dataset):
    def __init__(self, df):
        self.df = df
        self.tokenizer = tiktoken.get_encoding("r50k_base")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df[idx]["text"]
        tokens = self.tokenizer.encode(text)
        input_ids = torch.tensor(tokens[:-1])
        target = torch.tensor(tokens[1:])
        return input_ids, target

In [6]:
train_dataset=TinyStories(train_raw)
valid_dataset=TinyStories(valid_raw)

In [7]:
def collate_fn(batch):

    input_ids, target = zip(*batch)
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=PAD_ID)
    target = pad_sequence(target, batch_first=True, padding_value=PAD_ID)

    return input_ids, target


In [8]:
print(train_dataset.__getitem__(0))

(tensor([ 3198,  1110,    11,   257,  1310,  2576,  3706, 20037,  1043,   257,
        17598,   287,   607,  2119,    13,  1375,  2993,   340,   373,  2408,
          284,   711,   351,   340,   780,   340,   373,  7786,    13, 20037,
         2227,   284,  2648,   262, 17598,   351,   607,  1995,    11,   523,
          673,   714, 34249,   257,  4936,   319,   607, 10147,    13,   198,
          198,    43,   813,  1816,   284,   607,  1995,   290,   531,    11,
          366, 29252,    11,   314,  1043,   428, 17598,    13,  1680,   345,
         2648,   340,   351,   502,   290, 34249,   616, 10147,  1701,  2332,
         1995, 13541,   290,   531,    11,   366,  5297,    11, 20037,    11,
          356,   460,  2648,   262, 17598,   290,  4259,   534, 10147,   526,
          198,   198, 41631,    11,   484,  4888,   262, 17598,   290,   384,
        19103,   262,  4936,   319, 20037,   338, 10147,    13,   632,   373,
          407,  2408,   329,   606,   780,   484,   547,  7373,

In [9]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

test_dataloader = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [10]:
scaler = GradScaler(DEVICE)
import time


def train_step(model, optimizer, dataloader, epoch, device=DEVICE):
    t = time.perf_counter()
    model.train()
    train_loss = 4

    avg_loss = train_loss / len(dataloader)

    # progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}", dynamic_ncols=True)
    print(next(iter(dataloader)))
    for batch_idx, (inputs, target) in enumerate(dataloader):
        with autocast(device):
            print(inputs.shape)
            inputs = inputs.to(device)
            target = target.to(device)
            logits = model(inputs)
            loss = F.cross_entropy(logits.permute(0,2,1), target, ignore_index=50257, label_smoothing=0.1)

            batch_loss = loss.item()
            train_loss += batch_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        print("Avg Loss:", avg_loss)


        del inputs, target, loss, logits
        if batch_idx == 49:
            pb = (time.perf_counter()-t)/50
            print(f"{pb*1000:.0f} ms/batch, ~{pb*len(train_dataloader)/60:.0f} min/epoch")
            break

    avg_loss = train_loss / len(dataloader)

    print(f"Epoch {epoch} | Train Loss: {avg_loss:.4f}")
    torch.accelerator.empty_cache()
    return avg_loss



def test_step(model, epoch, dataloader, device=DEVICE):
    t = time.perf_counter()
    model.eval()
    test_loss, total_tokens = 0, 0

    avg_loss = 0.0

    with torch.no_grad():
        for batch_idx, (inputs, target) in dataloader:
            with autocast(device):
                inputs = inputs.to(device)
                target = target.to(device)
                logits = model(inputs)
                print(logits.shape)
                loss = F.cross_entropy(logits.permute(0,2,1), target, ignore_index=50257, label_smoothing=0.1, reduction='sum')

                batch_loss = loss.item()
                test_loss += batch_loss
                batch_tokens = (~inputs).sum().item()
                total_tokens += batch_tokens

            # batch_ppl = torch.exp(torch.tensor(batch_loss / batch_tokens))
            # progress_bar.set_postfix(batch_ppl=f"{batch_ppl:.4f}")

            del inputs, target, loss, logits
            if batch_idx == 49:
                pb = (time.perf_counter()-t)/50
                print(f"{pb*1000:.0f} ms/batch, ~{pb*len(train_dataloader)/60:.0f} min/epoch")
                break


    avg_loss = test_loss / total_tokens
    avg_nll = test_loss / total_tokens
    ppl = torch.exp(torch.tensor(avg_nll))

    print(f"Epoch {epoch} | Test Loss: {avg_loss:.4f} | Test Preplexity: {ppl:.4f}")
    torch.accelerator.empty_cache()
    return avg_loss, ppl

In [11]:
from transformer.transformer import Transformer

ChatGPT6 = Transformer(DEVICE)
ChatGPT6 = ChatGPT6.to(DEVICE)



In [12]:
current = 13
num_epochs = 1

start = 1 + current
epochs = num_epochs + current + 1

print(f"Running from {start} to {epochs-1}")

model = nn.DataParallel(ChatGPT6)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01, betas=(0.9, 0.95))

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=5,
    eta_min=3e-5
)

torch.manual_seed(42)
torch.cuda.manual_seed(42)

print("Ready to train!!")

for epoch in range(start, epochs):
    print("####### Epoch:", epoch)
    print("NOW: Training")

    train_step(
        model=model,
        optimizer=optimizer,
        epoch=epoch,
        dataloader=train_dataloader,
        device=DEVICE
    )

    print("NOW: Testing")
    test_step(
        model=model,
        epoch=epoch,
        dataloader=test_dataloader,
    )
    # scheduler.step()
    checkpoint = {
        'model_state_dict' : model.state_dict(),
        'optimizer_state_dict' : optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict()
    }
    torch.save(obj=checkpoint, f=f"BlockFormer_{epoch}_epochs.pth")

Running from 14 to 14
Ready to train!!
####### Epoch: 14
NOW: Training
(tensor([[   43,   813,   290,  ...,  3772, 50257, 50257],
        [ 7454,  2402,   257,  ..., 50257, 50257, 50257],
        [41083,   290,  5199,  ..., 50257, 50257, 50257],
        ...,
        [   41,   359,   550,  ..., 50257, 50257, 50257],
        [16305,   290,  3932,  ...,  1119,   547,  7926],
        [ 7454,  2402,   257,  ..., 50257, 50257, 50257]]), tensor([[  813,   290,  4186,  ...,    13, 50257, 50257],
        [ 2402,   257,   640,  ..., 50257, 50257, 50257],
        [  290,  5199,   547,  ..., 50257, 50257, 50257],
        ...,
        [  359,   550,   257,  ..., 50257, 50257, 50257],
        [  290,  3932,   547,  ...,   547,  7926,    13],
        [ 2402,   257,   640,  ..., 50257, 50257, 50257]]))
torch.Size([16, 840])


RuntimeError: permute(sparse_coo): number of dimensions in the tensor input does not match the length of the desired ordering of dimensions i.e. input.dim() = 2 is not equal to len(dims) = 3